# Image Classification — DINOv2 (Post Label Audit)

**Dataset:** Relabeled training set after the label audit pipeline detected and corrected mislabeled images in the `other` class.

**Model:** DINOv2 ViT-S/14 (frozen backbone) + lightweight classifier head.

---

## 1. Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support
)
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Data Overview (Post-Relabeling)

In [ ]:
TRAIN_DIR = './data_train'
TEST_DIR = './data_test'

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}\n')

def count_images(data_dir):
    counts = {}
    for cls in sorted(os.listdir(data_dir)):
        cls_path = os.path.join(data_dir, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([f for f in os.listdir(cls_path)
                               if f.lower().endswith(('.png','.jpg','.jpeg','.webp','.bmp'))])
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print('=== Train (relabeled) ===')
total_train = sum(train_counts.values())
for cls, cnt in train_counts.items():
    print(f'  {cls:30s}: {cnt:4d} ({cnt/total_train*100:.1f}%)')
print(f'  {"TOTAL":30s}: {total_train:4d}')

print(f'\n=== Test ===')
total_test = sum(test_counts.values())
for cls, cnt in test_counts.items():
    print(f'  {cls:30s}: {cnt:4d} ({cnt/total_test*100:.1f}%)')
print(f'  {"TOTAL":30s}: {total_test:4d}')

In [ ]:
# Distribution comparison
colors = sns.color_palette('husl', NUM_CLASSES)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, counts, title in [(axes[0], train_counts, 'Train (Relabeled)'), 
                           (axes[1], test_counts, 'Test')]:
    bars = ax.bar(counts.keys(), counts.values(), color=colors)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Images')
    ax.tick_params(axis='x', rotation=45)
    for bar, cnt in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                str(cnt), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('distribution_relabeled.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample images
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(20, 7))
fig.suptitle('Sample Images (Post-Relabeling)', fontsize=16, fontweight='bold')

for col, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(TRAIN_DIR, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.png','.jpg','.jpeg','.webp','.bmp'))]
    for row, fname in enumerate(random.sample(imgs, min(2, len(imgs)))):
        img = Image.open(os.path.join(cls_path, fname)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls.replace('_verified', ''), fontsize=9)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('samples_relabeled.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
test_dataset  = datasets.ImageFolder(root=TEST_DIR,  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {len(train_dataset)} images, {len(train_loader)} batches')
print(f'Test:  {len(test_dataset)} images, {len(test_loader)} batches')
print(f'Mapping: {train_dataset.class_to_idx}')

## 4. Model: DINOv2 + Classifier Head

In [ ]:
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
print(f'DINOv2 ViT-S loaded. Feature dim: {dinov2_backbone.embed_dim}')

In [ ]:
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, embed_dim=384):
        super().__init__()
        self.backbone = backbone
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone.eval()
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.classifier(features)
    
    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self


model = DINOv2Classifier(dinov2_backbone, NUM_CLASSES).to(device)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_p:>10,}')
print(f'Frozen (backbone): {total_p - train_p:>10,}')
print(f'Trainable (head):  {train_p:>10,} ({train_p/total_p*100:.2f}%)')

## 5. Feature Space Visualization (t-SNE)

Visualize how DINOv2 features separate the classes after relabeling. Compare with the previous t-SNE — `other` should now form a tighter, more distinct cluster.

In [ ]:
@torch.no_grad()
def extract_features(backbone, loader, device):
    backbone.eval()
    feats, labs = [], []
    for images, labels in loader:
        feats.append(backbone(images.to(device)).cpu().numpy())
        labs.extend(labels.numpy())
    return np.vstack(feats), np.array(labs)

print('Extracting features...')
train_features, train_labels = extract_features(dinov2_backbone, train_loader, device)
test_features, test_labels = extract_features(dinov2_backbone, test_loader, device)
print(f'Train: {train_features.shape}, Test: {test_features.shape}')

In [ ]:
all_feats = np.vstack([train_features, test_features])
all_labs = np.concatenate([train_labels, test_labels])
all_split = np.array(['train'] * len(train_labels) + ['test'] * len(test_labels))

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000)
emb = tsne.fit_transform(all_feats)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for idx, cls in enumerate(CLASS_NAMES):
    mask = all_labs == idx
    axes[0].scatter(emb[mask, 0], emb[mask, 1], c=[colors[idx]],
                    label=cls.replace('_verified', ''), alpha=0.7, s=50,
                    edgecolors='white', linewidth=0.5)
axes[0].set_title('t-SNE — by Class (Post-Relabeling)', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')

tm = all_split == 'train'; em = all_split == 'test'
axes[1].scatter(emb[tm, 0], emb[tm, 1], c='steelblue', label='Train', alpha=0.5, s=40)
axes[1].scatter(emb[em, 0], emb[em, 1], c='tomato', label='Test', alpha=0.8, s=60,
                edgecolors='black', linewidth=0.5)
axes[1].set_title('t-SNE — Train vs Test', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('tsne_relabeled.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0; correct = 0; total = 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'macro_f1': []}
best_f1 = 0.0; patience_counter = 0

print('=' * 85)
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | '
      f'{"Test Loss":>10} | {"Test Acc":>10} | {"Macro F1":>10} | {"LR":>10}')
print('=' * 85)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)
    macro_f1 = f1_score(labels, preds, average='macro')
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['macro_f1'].append(macro_f1)
    
    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>10.4f} | '
          f'{test_loss:>10.4f} | {test_acc:>10.4f} | {macro_f1:>10.4f} | {lr:>10.6f}')
    
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), 'best_model_relabeled.pth')
        patience_counter = 0
        print(f'         >>> New best (Macro F1: {best_f1:.4f})')
    else:
        patience_counter += 1
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nTraining complete. Best Macro F1: {best_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
er = range(1, len(history['train_loss']) + 1)

axes[0].plot(er, history['train_loss'], 'b-o', label='Train', ms=4)
axes[0].plot(er, history['test_loss'], 'r-o', label='Test', ms=4)
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(er, history['train_acc'], 'b-o', label='Train', ms=4)
axes[1].plot(er, history['test_acc'], 'r-o', label='Test', ms=4)
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(er, history['macro_f1'], 'g-o', label='Macro F1', ms=4)
axes[2].axhline(y=best_f1, color='green', ls='--', alpha=0.5, label=f'Best: {best_f1:.4f}')
axes[2].set_title('Macro F1', fontweight='bold'); axes[2].legend(); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curves_relabeled.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load('best_model_relabeled.pth', map_location=device))
test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

print(f'Test Accuracy: {test_acc:.4f}')
print(f'Macro F1:      {f1_score(all_labels, all_preds, average="macro"):.4f}')
print(f'Weighted F1:   {f1_score(all_labels, all_preds, average="weighted"):.4f}')

In [ ]:
print('\n' + '=' * 70)
print('CLASSIFICATION REPORT')
print('=' * 70)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Recall)', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrix_relabeled.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, average=None)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(NUM_CLASSES); w = 0.25
ax.bar(x - w, precision, w, label='Precision', color='#2196F3')
ax.bar(x, recall, w, label='Recall', color='#FF9800')
ax.bar(x + w, f1, w, label='F1-Score', color='#4CAF50')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_title('Per-Class Metrics', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_relabeled.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Misclassified examples
mis_idx = np.where(all_preds != all_labels)[0]
n_mis = len(mis_idx)
print(f'Misclassified: {n_mis} / {len(all_labels)} ({n_mis/len(all_labels)*100:.1f}%)')

if n_mis > 0:
    n_show = min(12, n_mis)
    samples = np.random.choice(mis_idx, size=n_show, replace=False)
    cols = min(6, n_show); rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.3*cols, 3.5*rows))
    axes_flat = np.array(axes).flatten()
    fig.suptitle('Misclassified Examples', fontsize=14, fontweight='bold')

    raw_ds = datasets.ImageFolder(root=TEST_DIR, transform=transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()]))

    for i, idx in enumerate(samples):
        ax = axes_flat[i]
        img, _ = raw_ds[idx]
        ax.imshow(img.permute(1, 2, 0).numpy())
        t = CLASS_NAMES[all_labels[idx]].replace('_verified', '')
        p = CLASS_NAMES[all_preds[idx]].replace('_verified', '')
        ax.set_title(f'True: {t}\nPred: {p}', fontsize=8, color='red'); ax.axis('off')
    for i in range(n_show, len(axes_flat)): axes_flat[i].axis('off')
    plt.tight_layout()
    plt.savefig('misclassified_relabeled.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No misclassifications — perfect accuracy!')

## 8. Test-Time Augmentation (TTA)

In [ ]:
tta_transforms = [
    test_transform,
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]

@torch.no_grad()
def evaluate_with_tta(model, test_dir, tta_transforms, device):
    model.eval()
    ds_list = [datasets.ImageFolder(root=test_dir, transform=t) for t in tta_transforms]
    loaders = [DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0) for ds in ds_list]
    
    all_logits = []
    for loader in loaders:
        batch_logits = []
        for images, _ in loader:
            batch_logits.append(model(images.to(device)).cpu())
        all_logits.append(torch.cat(batch_logits, dim=0))
    
    avg = torch.stack(all_logits).mean(dim=0)
    preds = avg.argmax(dim=1).numpy()
    labels = np.array(ds_list[0].targets)
    return (preds == labels).mean(), f1_score(labels, preds, average='macro'), preds, labels

tta_acc, tta_f1, tta_preds, tta_labels = evaluate_with_tta(model, TEST_DIR, tta_transforms, device)
base_f1 = f1_score(all_labels, all_preds, average='macro')

print(f'=== TTA Results ===')
print(f'  Accuracy: {tta_acc:.4f} (vs {test_acc:.4f})')
print(f'  Macro F1: {tta_f1:.4f} (vs {base_f1:.4f})')
print(f'\n{classification_report(tta_labels, tta_preds, target_names=CLASS_NAMES, digits=4)}')

## 9. Results Comparison

Compare all experiments across the project.

In [ ]:
# Fill in previous results for comparison
comparison = pd.DataFrame({
    'Experiment': [
        'EfficientNet-B0 + imbalance mitigation',
        'EfficientNet-B0 baseline',
        'DINOv2 (before relabeling)',
        'DINOv2 + CutMix/MixUp',
        'DINOv2 (after relabeling)',
        'DINOv2 (after relabeling) + TTA',
    ],
    'Macro F1': [
        0.7645,
        0.8088,
        0.8244,
        0.7896,
        best_f1,
        tta_f1,
    ],
    'Accuracy': [
        0.7667,
        0.7833,
        0.8167,
        0.7833,
        test_acc,
        tta_acc,
    ],
})

print('=' * 75)
print('FULL EXPERIMENT COMPARISON')
print('=' * 75)
print(comparison.to_string(index=False))
print('=' * 75)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison))
w = 0.35

ax.barh(x + w/2, comparison['Macro F1'], w, label='Macro F1', color='#4CAF50')
ax.barh(x - w/2, comparison['Accuracy'], w, label='Accuracy', color='#2196F3')

ax.set_yticks(x)
ax.set_yticklabels(comparison['Experiment'], fontsize=9)
ax.set_xlabel('Score')
ax.set_title('Experiment Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim(0.6, 1.0)
ax.grid(axis='x', alpha=0.3)

for i, (f1, acc) in enumerate(zip(comparison['Macro F1'], comparison['Accuracy'])):
    ax.text(f1 + 0.005, i + w/2, f'{f1:.4f}', va='center', fontsize=8)
    ax.text(acc + 0.005, i - w/2, f'{acc:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

### Full Project Timeline:

1. **EfficientNet-B0 + imbalance mitigation** (Macro F1: 0.7645) — weighted sampler + weighted loss hurt at 260 images
2. **EfficientNet-B0 baseline** (Macro F1: 0.8088) — removing mitigation improved results; 2.8x imbalance too mild
3. **Switched to DINOv2** (Macro F1: 0.8244) — self-supervised features dramatically better for few-shot
4. **CutMix/MixUp** (Macro F1: 0.7896) — augmentation noise hurt ambiguous classes, reverted
5. **Label audit** — t-SNE revealed `other` overlapping with `tu_hop`/`ngay_tet`; trained 5-class model to detect mislabeled images
6. **Retrained on relabeled data** — current results

### Key Insights:
- **Feature quality > training tricks** at small data scale
- **Label quality > class balance** — fixing mislabeled images had more impact than any model or augmentation change
- **Imbalance mitigation can be counterproductive** when the ratio is moderate and data is scarce
- **DINOv2 self-supervised features** are the right backbone for few-shot personal photo classification
- **Systematic label auditing** (train on clean classes, audit noisy classes) is a practical, scalable approach to data quality